# 03 · Modeling

Every guess and model is tested season by season with `src/evaluation.py`: for each of the last 9 seasons (2000–2009, no 2004), it may only learn from the seasons before, then predicts that season. The score is the average over the 9 seasons.

In [1]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd

from src.evaluation import test_seasons, evaluate_by_season, division_rank, mean_rank_error

team_table = pd.read_csv("../data/team_table.csv")
seasons = test_seasons(team_table)
seasons

[2000, 2001, 2002, 2003, 2005, 2006, 2007, 2008, 2009]

## Task (a): ranking

Score: on average, how many positions off the predicted rank is (0 = perfect).

### Simple guess: same order as last season

Inside each division, the team with the most points per game last season is predicted 1st, the next 2nd, and so on. New teams go last. This is "same position as last season" when divisions stay the same, and still works when they change (e.g. 1998, when the league went from 4 to 6 divisions).

In [2]:
def same_order_as_last_season(train, test):
    return division_rank(test, test["last_pts_per_game"])

simple_a = evaluate_by_season(team_table, same_order_as_last_season, mean_rank_error, seasons)
print("Average positions off:", round(simple_a["score"].mean(), 2))
simple_a

Average positions off: 1.07


,season,score
0,2000,0.666667
1,2001,1.200000
2,2002,1.066667
3,2003,1.000000
4,2005,1.133333
5,2006,1.133333
6,2007,1.466667
7,2008,0.800000
8,2009,1.200000


### For comparison: random order

What a guess with no information scores. Any model has to be clearly better than this, and better than the simple guess above.

In [3]:
rng = np.random.default_rng(0)

def random_order(train, test):
    return division_rank(test, rng.random(len(test)))

random_a = evaluate_by_season(team_table, random_order, mean_rank_error, seasons)
print("Average positions off:", round(random_a["score"].mean(), 2))

Average positions off: 1.55


## Task (b): coach change

Same loop: `evaluate_by_season(table, predict, score, seasons)`.

- `predict(train, test)` returns 1 (new coach) or 0 for each row of `test`, learning only from `train`.
- `score(test, predictions)` returns one number, e.g. the share of real coach changes caught (recall).